In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


CDC

In [1]:
spark.table("dim_customer").printSchema()
spark.sql("SELECT * FROM dim_customer LIMIT 5").show()

StatementMeta(, 9583538d-3d31-45ef-943c-070ce9d15640, 3, Finished, Available, Finished, False)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- customer_signup_date: date (nullable = true)
 |-- loyalty_points: integer (nullable = true)

+-----------+------------------+--------------------+-------+----------------+--------------------+--------------+
|customer_id|customer_full_name|               email|country|customer_segment|customer_signup_date|loyalty_points|
+-----------+------------------+--------------------+-------+----------------+--------------------+--------------+
|          9|   Carlos Arellano|  lori42@example.org|     in|         premium|          2021-07-15|         13086|
|         15|  Christopher Sims| bmartin@example.org|     in|         regular|          2023-08-25|         17780|
|         17|  Joseph Rodriguez|yfrederick@exampl...|     in|         premium|          2021-05-15|     

**<mark>Step 1 - Create a staged CDC dataset</mark>**

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

cdc_data = [
    (9,   "Carlos Arellano",   "carlos_new@example.org", "in", "vip",     date(2021, 7, 15), 15000),   # update
    (15,  "Christopher Sims",  "bmartin@example.org",    "in", "premium", date(2023, 8, 25), 19000),   # update
    (1000001, "Ava Sharma",    "ava.sharma@example.com", "in", "regular", date(2026, 3, 28), 500)      # insert
]

cdc_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_full_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("country", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("customer_signup_date", DateType(), True),
    StructField("loyalty_points", IntegerType(), True)
])

cdc_customer_df = spark.createDataFrame(cdc_data, schema=cdc_schema)
cdc_customer_df.show(truncate=False)

StatementMeta(, 9583538d-3d31-45ef-943c-070ce9d15640, 4, Finished, Available, Finished, False)

+-----------+------------------+----------------------+-------+----------------+--------------------+--------------+
|customer_id|customer_full_name|email                 |country|customer_segment|customer_signup_date|loyalty_points|
+-----------+------------------+----------------------+-------+----------------+--------------------+--------------+
|9          |Carlos Arellano   |carlos_new@example.org|in     |vip             |2021-07-15          |15000         |
|15         |Christopher Sims  |bmartin@example.org   |in     |premium         |2023-08-25          |19000         |
|1000001    |Ava Sharma        |ava.sharma@example.com|in     |regular         |2026-03-28          |500           |
+-----------+------------------+----------------------+-------+----------------+--------------------+--------------+



###### **<mark>Step 2 - Create a temp view for MERGE</mark>**

In [3]:
cdc_customer_df.createOrReplaceTempView("stg_customer_cdc")

StatementMeta(, 9583538d-3d31-45ef-943c-070ce9d15640, 5, Finished, Available, Finished, False)

###### **<mark>Step 3 - Apply CDC using MERGE</mark>**

In [4]:
spark.sql("""
MERGE INTO dim_customer AS target
USING stg_customer_cdc AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
UPDATE SET
    target.customer_full_name = source.customer_full_name,
    target.email = source.email,
    target.country = source.country,
    target.customer_segment = source.customer_segment,
    target.customer_signup_date = source.customer_signup_date,
    target.loyalty_points = source.loyalty_points

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_full_name,
    email,
    country,
    customer_segment,
    customer_signup_date,
    loyalty_points
)
VALUES (
    source.customer_id,
    source.customer_full_name,
    source.email,
    source.country,
    source.customer_segment,
    source.customer_signup_date,
    source.loyalty_points
)
""")

StatementMeta(, 9583538d-3d31-45ef-943c-070ce9d15640, 6, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

###### **<mark>Step 4 - Validate CDC Results</mark>**

In [5]:
spark.sql("""
SELECT *
FROM dim_customer
WHERE customer_id IN (9, 15, 1000001)
ORDER BY customer_id
""").show(truncate=False)

StatementMeta(, 9583538d-3d31-45ef-943c-070ce9d15640, 7, Finished, Available, Finished, False)

+-----------+------------------+----------------------+-------+----------------+--------------------+--------------+
|customer_id|customer_full_name|email                 |country|customer_segment|customer_signup_date|loyalty_points|
+-----------+------------------+----------------------+-------+----------------+--------------------+--------------+
|9          |Carlos Arellano   |carlos_new@example.org|in     |vip             |2021-07-15          |15000         |
|15         |Christopher Sims  |bmartin@example.org   |in     |premium         |2023-08-25          |19000         |
|1000001    |Ava Sharma        |ava.sharma@example.com|in     |regular         |2026-03-28          |500           |
+-----------+------------------+----------------------+-------+----------------+--------------------+--------------+



In [6]:
spark.sql("SELECT COUNT(*) FROM dim_customer").show()

StatementMeta(, 9583538d-3d31-45ef-943c-070ce9d15640, 8, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
| 1000001|
+--------+

